# Consolidação de setembro e outubro: duas janelas

Etapa posterior ao experimento de qualidade. V1 exclui somente fins de semana; mantém dias úteis suspeitos. Não reexecuta experimentos históricos, não ajusta modelos e não substitui previsões enviadas. Setembro continua retrospectivo com alvos ocultos; outubro usa data real de emissão. A função é idempotente: se os arquivos já existem, apenas confere hashes. Nenhuma publicação no GitHub nesta etapa.

In [1]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
ROOT = Path(r'F:\PRODUTO_MESTRADO')
run = subprocess.run([sys.executable, str(ROOT/'scripts/consolidar_previsoes_qualificacao.py')], capture_output=True, text=True, encoding='utf-8', check=True)
print(run.stdout)

{
  "status": "Emissões existentes conferidas, sem reemissão",
  "pasta_setembro": "F:\\PRODUTO_MESTRADO\\resultados\\validacao_setembro_2026\\suplemento_V1_20261004",
  "pasta_outubro": "F:\\PRODUTO_MESTRADO\\resultados\\previsao_antecipada_compras\\emissoes\\20261004_073829",
  "setembro_datas_por_janela": 20,
  "outubro_previsoes": 5,
  "metricas_2026_calculadas": false
}



## Setembro: cortes simulados, consumos desconhecidos

São 20 datas elegíveis por janela (40 previsões válidas); as 60 linhas incluem exclusões, que permanecem sem previsão. Não atribuir antecedência operacional real às origens simuladas. O quantitativo institucional é somente comparador posterior, pois o recebimento em cada origem não está documentado. Pura, +10 e P90 permanecem separados.

In [2]:
folder = ROOT/'resultados/validacao_setembro_2026/suplemento_V1_20261004'
september = pd.read_csv(folder/'previsoes_setembro_V1_congeladas.csv', parse_dates=['data','origem_simulada'])
assert september.consumo_observado.isna().all()
assert september.loc[~september.avaliacao_elegivel,'plano_puro'].isna().all()
display(september.groupby('janela').agg(linhas=('data','size'), datas_elegiveis=('avaliacao_elegivel','sum')))
display(september.loc[september.avaliacao_elegivel, ['data','janela','origem_simulada','ultima_data_consumo_historico','plano_puro','plano_moderado','plano_conservador']])

,linhas,datas_elegiveis
janela,,
Compras,30,20
Preparo,30,20


,data,janela,origem_simulada,ultima_data_consumo_historico,plano_puro,plano_moderado,plano_conservador
0,2026-09-01,Preparo,2026-08-28,2026-08-27,175.0,185.0,212.0
1,2026-09-02,Preparo,2026-08-28,2026-08-27,175.0,185.0,213.0
2,2026-09-03,Preparo,2026-08-28,2026-08-27,163.0,173.0,200.0
3,2026-09-04,Preparo,2026-08-28,2026-08-27,161.0,171.0,198.0
8,2026-09-09,Preparo,2026-09-04,2026-08-31,176.0,186.0,213.0
9,2026-09-10,Preparo,2026-09-04,2026-08-31,169.0,179.0,206.0
10,2026-09-11,Preparo,2026-09-04,2026-08-31,182.0,192.0,219.0
13,2026-09-14,Preparo,2026-09-11,2026-08-31,186.0,196.0,223.0
14,2026-09-15,Preparo,2026-09-11,2026-08-31,174.0,184.0,211.0
15,2026-09-16,Preparo,2026-09-11,2026-08-31,175.0,185.0,212.0


## Outubro: preservar a emissão anterior

A nova versão é de domingo, 04/10, com antecedência real de 1–5 dias, mesmo usando a sexta 02/10 como corte das informações. O combinado curto já contém +10; não recebe outra margem. A semana de 12–16/10 não tem refeições. Compras 19–23/10 serão emitidas em 09/10, com atualização para preparo em 16/10; nenhuma dessas duas execuções futuras é antecipada aqui.

In [3]:
report = ROOT/'resultados/acompanhamento_duas_janelas_20261004'
october = pd.read_csv(report/'previsoes_preparo_05_09_outubro_V1.csv', parse_dates=['data'])
assert len(october)==5 and october.janela.eq('Preparo').all()
assert october.consumo_observado.isna().all()
assert october.antecedencia_real_dias.tolist()==[1,2,3,4,5]
display(october[['data','quantitativo_institucional','plano_puro','plano_moderado','plano_conservador','plano_combinado_mais10','emitido_em','antecedencia_real_dias']])
display(pd.read_csv(report/'cronograma_outubro.csv'))
print('Sem pontuação enquanto os consumos estiverem ausentes. Nenhuma alegação de superioridade operacional.')

,data,quantitativo_institucional,plano_puro,plano_moderado,plano_conservador,plano_combinado_mais10,emitido_em,antecedencia_real_dias
0,2026-10-05,225,178,188,215,224,2026-10-04T07:38:29.628851-03:00,1
1,2026-10-06,220,167,177,204,217,2026-10-04T07:38:29.628851-03:00,2
2,2026-10-07,220,169,179,207,218,2026-10-04T07:38:29.628851-03:00,3
3,2026-10-08,210,161,171,198,208,2026-10-04T07:38:29.628851-03:00,4
4,2026-10-09,200,167,177,204,202,2026-10-04T07:38:29.628851-03:00,5


,refeicoes,origem_nominal,finalidade,situacao
0,05–09/10/2026,02/10/2026,Preparo,Nova emissão V1 real em 04/10; original preser...
1,12–16/10/2026,02/10/2026,Compras,"Sem emissão: feriado/recesso, sem refeições; n..."
2,19–23/10/2026,09/10/2026,Compras,"Execução futura; 10–14 dias antes, sem quantit..."
3,19–23/10/2026,16/10/2026,Preparo,Execução futura; atualizar previsão anterior e...


Sem pontuação enquanto os consumos estiverem ausentes. Nenhuma alegação de superioridade operacional.


## Integridade

Os modelos permanecem ajustados em 2023–2025. Consumos anteriores de 2026 apenas compõem o histórico disponível antes de cada origem. As margens usam erros próprios de 2023–2025. A verificação de invariância não encontrou mudança dos preditores quando alvos futuros e reservas foram perturbados. A disponibilidade histórica dos demais arquivos permanece declarada, sem comprovação de cada recebimento.

In [4]:
from hashlib import sha256
manifest = json.loads((report/'manifesto_consolidacao.json').read_text(encoding='utf-8'))
for rel, expected in manifest['arquivos_sha256'].items():
    assert sha256((ROOT/rel).read_bytes()).hexdigest()==expected, rel
for rel, expected in manifest['originais_preservados_sha256'].items():
    assert sha256((ROOT/rel).read_bytes()).hexdigest()==expected, rel
print('Arquivos congelados e originais conferidos por SHA-256.')

Arquivos congelados e originais conferidos por SHA-256.
